# Healthcare ML – Test Results Classification
**Course:** CDS3533  
**Instructor:** Dr. Abdelrhman Rezkallah  

Applying Supervised Learning (Classification) to predict **Test Results** (Normal / Abnormal / Inconclusive / Pending) from patient admission data.

This notebook follows the same pipeline used in the Titanic Streamlit deployment:
1. Load & Explore
2. EDA (visualisations)
3. Feature Engineering
4. Train / Test Split
5. Feature Selection
6. Multiple Classifiers
7. Cross Validation
8. Hyper-parameter Tuning
9. Final Pipeline → `healthcare.pkl`
10. Streamlit App

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Load Dataset

In [ ]:
df = pd.read_csv('healthcare_cleaned.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

## 3. Feature Engineering

In [ ]:
# Parse dates and compute Length of Stay
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date']    = pd.to_datetime(df['Discharge Date'])
df['Length of Stay']    = (df['Discharge Date'] - df['Date of Admission']).dt.days

# Keep only useful columns
df = df[['Age','Gender','Blood Type','Medical Condition',
         'Insurance Provider','Billing Amount','Admission Type',
         'Medication','Length of Stay','Test Results']]
df.head()

## 4. Exploratory Data Analysis

In [ ]:
# Target distribution
plt.figure(figsize=[8,4])
sns.countplot(x='Test Results', data=df, palette='Set2')
plt.title('Test Results Distribution')
plt.show()

In [ ]:
sns.countplot(x='Medical Condition', hue='Test Results', data=df, palette='husl')
plt.xticks(rotation=45)
plt.title('Medical Condition vs Test Results')
plt.show()

In [ ]:
sns.countplot(x='Admission Type', hue='Test Results', data=df, palette='Set1')
plt.title('Admission Type vs Test Results')
plt.show()

In [ ]:
sns.countplot(x='Gender', hue='Test Results', data=df, palette='pastel')
plt.title('Gender vs Test Results')
plt.show()

In [ ]:
sns.histplot(df['Age'], bins=20, kde=True, color='steelblue')
plt.title('Age Distribution')
plt.show()

In [ ]:
sns.boxplot(x='Test Results', y='Billing Amount', data=df, palette='Set3')
plt.title('Billing Amount by Test Results')
plt.show()

In [ ]:
sns.boxplot(x='Test Results', y='Age', data=df, palette='Set2')
plt.title('Age by Test Results')
plt.show()

## 5. Split Input / Output

In [ ]:
X = df.drop('Test Results', axis=1)
y = df['Test Results']

## 6. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 7. Feature Selection (Mutual Information)

In [ ]:
# Encode for feature selection scoring
x_enc = pd.get_dummies(X, drop_first=True)

from sklearn.feature_selection import mutual_info_classif, SelectKBest
fs = SelectKBest(mutual_info_classif, k='all')
fs.fit(x_enc, y)

plt.figure(figsize=[14,5])
sns.barplot(x=list(fs.feature_names_in_), y=fs.scores_, palette='husl')
plt.xticks(rotation=90)
plt.title('Feature Importance (Mutual Information)')
plt.tight_layout()
plt.show()

## 8. Pipeline Setup

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

numeric_columns = x_train.select_dtypes(exclude='object').columns
cat_columns     = x_train.select_dtypes(include='object').columns

print('Numeric:', list(numeric_columns))
print('Categorical:', list(cat_columns))

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ('handle missing value', SimpleImputer(strategy='median')),
    ('scaling', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('handle missing value', SimpleImputer(strategy='most_frequent')),
    ('one hot encoder', OneHotEncoder(drop='first', handle_unknown='ignore')),
    ('scaling', StandardScaler(with_mean=False))
])

preprocessing = ColumnTransformer(transformers=[
    ('numerical_columns', numerical_pipeline, numeric_columns),
    ('cat_columns', cat_pipeline, cat_columns)
], remainder='passthrough')

## 9. Multiple Classifiers

In [ ]:
from sklearn.metrics import classification_report

# Helper: build pipeline with any model
def evaluate_model(model, name):
    pipe = Pipeline(steps=[('preprocessing', preprocessing), ('modeling', model)])
    pipe.fit(x_train, y_train)
    y_pred = pipe.predict(x_test)
    print(f'\n===== {name} =====')
    print('TRAIN')
    print(classification_report(y_train, pipe.predict(x_train)))
    print('TEST')
    print(classification_report(y_test, y_pred))
    return pipe

In [ ]:
from sklearn.linear_model import LogisticRegression
evaluate_model(LogisticRegression(max_iter=1000), 'Logistic Regression');

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
evaluate_model(KNeighborsClassifier(), 'KNN');

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline as P
# GNB needs dense input – rebuild without sparse scaler
cat_pipeline_gnb = Pipeline(steps=[
    ('handle missing value', SimpleImputer(strategy='most_frequent')),
    ('one hot encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])
prep_gnb = ColumnTransformer(transformers=[
    ('numerical_columns', numerical_pipeline, numeric_columns),
    ('cat_columns', cat_pipeline_gnb, cat_columns)
], remainder='passthrough')
gnb_pipe = Pipeline(steps=[('preprocessing', prep_gnb), ('modeling', GaussianNB())])
gnb_pipe.fit(x_train, y_train)
y_pred_gnb = gnb_pipe.predict(x_test)
print('\n===== Naive Bayes =====')
print('TEST')
print(classification_report(y_test, y_pred_gnb))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
evaluate_model(DecisionTreeClassifier(), 'Decision Tree');

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_pipe = evaluate_model(RandomForestClassifier(random_state=42), 'Random Forest')

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

xgb_pipe = Pipeline(steps=[('preprocessing', preprocessing),
                            ('modeling', XGBClassifier(eval_metric='mlogloss', random_state=42))])
xgb_pipe.fit(x_train, y_train_enc)
y_pred_xgb = xgb_pipe.predict(x_test)
print('\n===== XGBoost =====')
print(classification_report(y_test_enc, y_pred_xgb, target_names=le.classes_))

## 10. Cross Validation (Random Forest)

In [ ]:
from sklearn.model_selection import KFold, cross_validate

kfold  = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(rf_pipe, x_train, y_train, cv=kfold)

print('CV scores:', scores['test_score'])
print('Mean CV accuracy:', scores['test_score'].mean().round(4))

## 11. Hyper-parameter Tuning (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {
    'modeling__n_estimators': [100, 200, 300],
    'modeling__max_depth':    [5, 7, 9]
}
grid_search = GridSearchCV(rf_pipe, param_grid=parameters, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(x_train, y_train)

In [ ]:
print('Best params:', grid_search.best_params_)

## 12. Final Pipeline + Save Model

In [ ]:
best = grid_search.best_params_

final_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('modeling', RandomForestClassifier(
        n_estimators=best['modeling__n_estimators'],
        max_depth=best['modeling__max_depth'],
        random_state=42
    ))
])

final_pipeline.fit(x_train, y_train)
y_pred_final = final_pipeline.predict(x_test)
print('TRAIN')
print(classification_report(y_train, final_pipeline.predict(x_train)))
print('TEST')
print(classification_report(y_test, y_pred_final))

In [ ]:
import joblib
joblib.dump(final_pipeline, 'healthcare.pkl')
print('Model saved as healthcare.pkl')

## 13. Test Single Prediction

In [ ]:
classifier = joblib.load('healthcare.pkl')

sample = pd.DataFrame({
    'Age':                [45],
    'Gender':             ['Female'],
    'Blood Type':         ['A+'],
    'Medical Condition':  ['Diabetes'],
    'Insurance Provider': ['Aetna'],
    'Billing Amount':     [15000.0],
    'Admission Type':     ['Elective'],
    'Medication':         ['Metformin'],
    'Length of Stay':     [5]
})

prediction = classifier.predict(sample)
print('Predicted Test Result:', prediction[0])

## 14. Deploy with Streamlit
Run the cells below to launch the app via ngrok (same approach as the Titanic notebook).

In [ ]:
! pip install pyngrok streamlit

In [ ]:
from pyngrok import ngrok
ngrok.kill()

NGROK_AUTH_TOKEN = 'YOUR_NGROK_TOKEN_HERE'   # ← paste your token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.connect(8501)

In [ ]:
%%writefile healthcare_app.py
import pandas as pd
import joblib
import streamlit as st

classifier = joblib.load('healthcare.pkl')

def predict_test_result(age, gender, blood_type, condition,
                        insurance, billing, admission_type,
                        medication, length_of_stay):
    sample = pd.DataFrame({
        'Age':                [age],
        'Gender':             [gender],
        'Blood Type':         [blood_type],
        'Medical Condition':  [condition],
        'Insurance Provider': [insurance],
        'Billing Amount':     [billing],
        'Admission Type':     [admission_type],
        'Medication':         [medication],
        'Length of Stay':     [length_of_stay]
    })
    return classifier.predict(sample)[0]

def main():
    st.title('🏥 Healthcare Test Result Prediction')
    html_temp = """
        <div style="background-color:#1f77b4;padding:10px">
        <h2 style="color:white;text-align:center;">Patient Test Result Classifier</h2>
        </div>
    """
    st.markdown(html_temp, unsafe_allow_html=True)
    st.markdown("---")

    col1, col2 = st.columns(2)

    with col1:
        age            = st.slider('Age', 0, 100, 35)
        gender         = st.radio('Gender', ['Male', 'Female', 'Other'])
        blood_type     = st.selectbox('Blood Type', ['A+','A-','B+','B-','O+','O-','AB+','AB-'])
        condition      = st.selectbox('Medical Condition', ['Cancer','Obesity','Diabetes','Asthma',
                                                             'Hypertension','Arthritis','Heart Disease',
                                                             'COVID-19','Flu','Migraine'])
        insurance      = st.selectbox('Insurance Provider', ['Blue Cross','Medicare','Aetna',
                                                              'UnitedHealthcare','Cigna','Unknown'])

    with col2:
        billing        = st.number_input('Billing Amount ($)', 0.0, 40000.0, 10000.0, step=500.0)
        admission_type = st.radio('Admission Type', ['Urgent', 'Emergency', 'Elective'])
        medication     = st.selectbox('Medication', ['Paracetamol','Ibuprofen','Aspirin',
                                                      'Penicillin','Lipitor','Metformin',
                                                      'Albuterol','Lisinopril','Atorvastatin'])
        length_of_stay = st.slider('Length of Stay (days)', 0, 60, 5)

    st.markdown("---")
    result = ''
    if st.button('🔍 Predict Test Result'):
        result = predict_test_result(age, gender, blood_type, condition,
                                     insurance, billing, admission_type,
                                     medication, length_of_stay)
        color = {'Normal':'green','Abnormal':'red',
                 'Inconclusive':'orange','Pending':'blue'}.get(result,'gray')
        st.markdown(f"<h3 style='color:{color};'>Predicted Test Result: {result}</h3>",
                    unsafe_allow_html=True)

if __name__ == '__main__':
    main()

In [ ]:
! streamlit run healthcare_app.py